In [1]:
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../challenge_data/vm_cpu_readings.csv.gz')

In [3]:
g = df.groupby("timestamp")
active = g["vm_id"].nunique()
agg_cpu = g["avg_cpu"].mean()

In [4]:
train_cpu, hold_cpu = agg_cpu.iloc[:110], agg_cpu.iloc[110:]

df = pd.DataFrame({"unique_id": "agg_cpu",
                      "ds": np.arange(len(train_cpu)),
                      "y": train_cpu.values})

In [5]:
sf_arima = StatsForecast(
    models=[
        ARIMA(order=(3, 0, 0), season_length=1, alias="AR3"),   # your PACF-based choice
        AutoARIMA(season_length=1, alias="AutoARIMA"),           # picks (p, d, q) itself, for comparison
    ],
    freq=1,
)

In [7]:
cv_arima = sf_arima.cross_validation(df=df, h=24, step_size=4, n_windows=10)
print(sorted(cv_arima["cutoff"].unique()))   # should be 49, 53, ..., 85
print(cv_arima["ds"].max())                  

[np.int64(49), np.int64(53), np.int64(57), np.int64(61), np.int64(65), np.int64(69), np.int64(73), np.int64(77), np.int64(81), np.int64(85)]
109


In [10]:
e = cv_arima["y"] - cv_arima["AR3"]
print("MAE", e.abs().mean(), "RMSE", (e**2).mean() ** 0.5)

MAE 0.19184671173402593 RMSE 0.23062699673807927


In [12]:
from statsforecast.models import Naive, HistoricAverage

sf = StatsForecast(
    models=[Naive(), HistoricAverage(), ARIMA(order=(3, 0, 0), season_length=1, alias="AR3")],
    freq=1,
)
cv = sf.cross_validation(df=df, h=24, step_size=4, n_windows=10)
models = ["Naive", "HistoricAverage", "AR3"]

# per fold: RMSE for each model, and how many folds AR3 wins
fold = (cv.assign(**{m: (cv["y"] - cv[m])**2 for m in models})
          .groupby("cutoff")[models].mean() ** 0.5)
print(fold.round(3))
print("AR3 beats HistoricAverage in", (fold["AR3"] < fold["HistoricAverage"]).sum(), "of 10 folds")
print("AR3 beats Naive in", (fold["AR3"] < fold["Naive"]).sum(), "of 10 folds")

# per horizon step
cv["step"] = cv.groupby("cutoff").cumcount() + 1
print(cv.assign(**{m: (cv["y"] - cv[m]).abs() for m in models})
        .groupby("step")[models].mean().round(3))

        Naive  HistoricAverage    AR3
cutoff                               
49      0.303            0.337  0.337
53      0.202            0.335  0.337
57      0.307            0.271  0.181
61      0.346            0.210  0.162
65      0.496            0.178  0.213
69      0.333            0.177  0.250
73      0.152            0.193  0.197
77      0.273            0.190  0.162
81      0.167            0.205  0.182
85      0.154            0.233  0.200
AR3 beats HistoricAverage in 6 of 10 folds
AR3 beats Naive in 5 of 10 folds
      Naive  HistoricAverage    AR3
step                               
1     0.098            0.197  0.102
2     0.131            0.241  0.154
3     0.162            0.229  0.158
4     0.131            0.206  0.143
5     0.182            0.223  0.192
6     0.227            0.281  0.243
7     0.216            0.229  0.199
8     0.154            0.195  0.170
9     0.249            0.228  0.224
10    0.262            0.249  0.243
11    0.220            0.207  0.202


AR(3) has the lowest pooled MAE and RMSE, the lowest mean fold RMSE, and ties for the lowest worst-case fold.
It beats HistoricAverage in 6 of 10 folds and Naive in 5 of 10, on heavily overlapping folds. That’s not strong evidence of a real difference between AR(3) and HistoricAverage.
No model wins everywhere. The best model depends on the horizon: Naive for the first few steps, HistoricAverage for the last 12.